# Initialization & Data Cleaning

## Import libraries

In [17]:
import pandas as pd
import numpy as np
from pathlib import Path


## Load all monthly data

In [18]:

data_path = Path(".")
files = list(data_path.glob("*.csv"))

dfs = []
for file in files:
    df = pd.read_csv(file, sep="\t")
    df["source_file"] = file.name
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)

print("Initial shape:", data.shape)
print("Columns:", data.columns)


Initial shape: (240447, 9)
Columns: Index(['sensor_id', 'sensor_type', 'location', 'lat', 'lon', 'timestamp',
       'value_type', 'value', 'source_file'],
      dtype='object')


##  Basic Cleaning

### Convert timestamp

In [19]:
data["timestamp"] = pd.to_datetime(data["timestamp"], errors="coerce")

### Drop critical missing values

In [20]:
data = data.dropna(subset=["timestamp", "lat", "lon", "value"])

### Remove Duplicates

In [21]:
data = data.drop_duplicates()

In [22]:
print("After basic cleaning:", data.shape)


After basic cleaning: (240447, 9)


### Convert to wide format

In [24]:
wide_df = (
    data
    .pivot_table(
        index=["sensor_id","location","lat","lon","timestamp"],
        columns="value_type",
        values="value",
        aggfunc="mean"
    )
    .reset_index()
)

print("After pivot:", wide_df.shape)
print(wide_df.columns)


After pivot: (94679, 10)
Index(['sensor_id', 'location', 'lat', 'lon', 'timestamp', 'P0', 'P1', 'P2',
       'humidity', 'temperature'],
      dtype='object', name='value_type')


### Rename columns

In [31]:
wide_df = wide_df.rename(columns={
    "P0": "PM1",
    "P1": "PM2_5",
    "P2": "PM10"
})

print(wide_df.columns)


Index(['sensor_id', 'location', 'lat', 'lon', 'timestamp', 'PM1', 'PM2_5',
       'PM10', 'humidity', 'temperature', 'date'],
      dtype='object', name='value_type')


### Handle missing values

In [33]:
wide_df = wide_df.sort_values(["sensor_id", "timestamp"])

feature_cols = ["PM1","PM2_5","PM10","humidity","temperature"]

# Forward fill per sensor
wide_df[feature_cols] = (
    wide_df
    .groupby("sensor_id")[feature_cols]
    .ffill()
)

# Sensor-wise median
for col in feature_cols:
    wide_df[col] = wide_df[col].fillna(
        wide_df.groupby("sensor_id")[col].transform("median")
    )

# Global fallback
wide_df[feature_cols] = wide_df[feature_cols].fillna(
    wide_df[feature_cols].median()
)

print("Remaining missing values:")
print(wide_df[feature_cols].isna().sum())


Remaining missing values:
value_type
PM1            0
PM2_5          0
PM10           0
humidity       0
temperature    0
dtype: int64


### Remove extreme outliers

In [34]:
# Remove physically impossible values
wide_df = wide_df[
    (wide_df["PM1"] >= 0) &
    (wide_df["PM2_5"] >= 0) &
    (wide_df["PM10"] >= 0)
]

# Clip extreme upper values ( in the 99th percentile )
for col in ["PM1","PM2_5","PM10"]:
    upper = wide_df[col].quantile(0.99)
    wide_df[col] = np.clip(wide_df[col], None, upper)



### * Daily Aggregation

In [35]:
wide_df["date"] = wide_df["timestamp"].dt.date

daily_df = (
    wide_df
    .groupby(["sensor_id","location","lat","lon","date"])[feature_cols]
    .mean()
    .reset_index()
)

print("Daily aggregated shape:", daily_df.shape)


Daily aggregated shape: (641, 10)


### Current dataset

In [36]:
print(daily_df.head(25))

value_type  sensor_id  location    lat  lon        date        PM1  \
0                4851      3627  6.515  3.4  2024-02-01  37.745000   
1                4851      3627  6.515  3.4  2024-02-07  59.082457   
2                4851      3627  6.515  3.4  2024-02-19  25.237632   
3                4851      3627  6.515  3.4  2024-02-21  34.854342   
4                4851      3627  6.515  3.4  2024-02-23  40.307029   
5                4851      3627  6.515  3.4  2024-02-25  30.744537   
6                4851      3627  6.515  3.4  2024-02-27  21.625713   
7                4851      3627  6.515  3.4  2024-02-28  20.668955   
8                4851      3627  6.515  3.4  2024-02-29  18.275918   
9                4851      3627  6.515  3.4  2024-04-02  12.867768   
10               4851      3627  6.515  3.4  2024-04-04  12.331155   
11               4851      3627  6.515  3.4  2024-04-05  17.482821   
12               4851      3627  6.515  3.4  2024-04-06  10.061712   
13               485